In [ ]:
# Paths (added for the public repo). Paths below are relative to this notebook;
# override with the DATA_ROOT / RUNS_ROOT environment variables.
import os
DATA_ROOT = os.environ.get('DATA_ROOT', '../../data')   # data/blouse, data/aihub_official
RUNS_ROOT = os.environ.get('RUNS_ROOT', '../../runs')   # checkpoints / metrics are written here


# [TUNED] Plain ResNet-50 베이스라인 — 과적합 완화 코드 튜닝판

원본 `baseline_B0_local.ipynb`에서 **새 데이터 없이** 적용 가능한 항목만 수정한 버전입니다.

**변경 요약 (모두 `[TUNED]` 주석으로 표시)**

1. **Early stopping** — `val_F1`이 `early_stop_patience`(기본 4)에폭 연속 개선되지 않으면 학습 중단. 원본은 val 정점(≈epoch 10) 이후 10에폭을 낭비했음.
2. **이중 불균형 보정 정리** — 원본은 `WeightedRandomSampler` + Loss `class_weight`를 동시에 사용(이중 보정). `cfg.imbalance_strategy`로 **하나만** 선택 (기본 `'class_weight'`).
3. **head Dropout 추가** — `cfg.dropout`(기본 0.3)로 head 앞에 Dropout 삽입.
4. **weight_decay 강화** — `1e-4 → 1e-3`.

결과는 원본을 덮어쓰지 않도록 `baseline_resnet50_tuned` 폴더에 저장됩니다. 모델 구조(Plain ResNet-50 단일 헤드)는 그대로라 B0 베이스라인 성격을 유지합니다.

In [ ]:
# (로컬 실행) Google Drive 마운트 불필요 — 실행해도 아무 동작 안 합니다.
print('local run: drive mount skipped')


In [ ]:
# (로컬 실행) 데이터는 이미 로컬에 있으므로 unzip 불필요.
print('local run: using', f'{DATA_ROOT}/blouse')


In [ ]:
from pathlib import Path

root = Path(f"{DATA_ROOT}/blouse")
print(root.exists())
print(len(list(root.rglob("*.jpg"))), "jpg")
print(len(list(root.rglob("*.png"))), "png")
print(len(list(root.rglob("*.json"))), "json")
print([p.name for p in root.iterdir()][:10])

# 폐의류 재활용 분류 — Plain ResNet 베이스라인 (Colab)

본 노트북은 프로젝트 제안서의 **Ablation Study** 출발점인
`Baseline (Plain ResNet)` 단계를 구현합니다.

**Strict Baseline 정의**
- 백본: ImageNet 사전학습된 **ResNet-50** (수정 없음, Plain)
- 헤드: **단일 헤드 (Main Task: 의류 상태 3-class)**
- 손실: CrossEntropyLoss (+ class weight, '재사용' 다수 클래스 보정)
- 입력 해상도: 224×224
- 최적화: AdamW + CosineAnnealing, AMP(혼합정밀도) 사용

이 노트북에서 산출되는 수치(F1-macro, Confusion Matrix, Params/FLOPs/FPS)는
이후 단계(Texture-aware Block 추가 → Multi-Task Head → DenseNet 백본 교체)와
비교하기 위한 **고정 기준점**입니다.

> ⚙️ Multi-Task로 확장하려면 모델 / Loss / 학습 루프에 주석 `# [MULTI-TASK]` 가
> 붙은 부분만 풀면 됩니다.

**환경 가정**: Colab Free (T4 16GB) 또는 Kaggle.


## 0. 환경 설치
- 준비물 1. thop 설치
- seaborn, scikit-learn 라이브러리 설치
- 성능계산(FLOPs) 및 시각화(혼동행렬), 평가지표 계산에 필요합니다.

In [ ]:
# Colab 기본 환경에는 대부분 설치되어 있으나, FLOPs 측정용 thop 만 추가 설치
!pip -q install thop seaborn scikit-learn


## 1. Imports & 재현성 시드
- 기본 설정 및 시드 고정
- 파이토치, 넘파이 등 필요한 라이브러리 불러오고 seed고정

In [ ]:
import os, json, random, time, math, copy
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, accuracy_score,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '| Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # 속도 우선
    torch.backends.cudnn.benchmark = True

seed_everything(42)


# === [TRUNCATED IMAGE FIX] 잘린(손상) 이미지도 읽도록 허용 ===
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 2. 설정 (Config)

- 실험 변수는 여기에만 두어 Ablation 비교를 용이하게 합니다.
- 데이터 위치(data_root), 결과 저장 위치(output_dir)를 정합니다.
- 배치 크기, epoch, 학습률 같은 하이퍼파라미터를 한곳에서 관리합니다.

In [ ]:
@dataclass  #config라는 설정 묶음을 깔끔하게 만들기 위함
class Config: #설정 값을 모아놓은 클래스
    # 경로 — 본인 환경에 맞게 수정
    # ⚠️ 위에서 find 명령으로 찾은 '01.원천데이터'/'02.라벨링데이터'의 공통 부모 폴더
    data_root: str = f'{DATA_ROOT}/blouse'    #drive 내에 있는 원본 데이터
    # [TUNED] 원본 베이스라인(baseline_resnet50)을 덮어쓰지 않도록 별도 폴더에 저장
    output_dir: str = f'{RUNS_ROOT}/baseline_resnet50_tuned'

    # 클래스
    main_classes: tuple = ('재사용', '재활용', '폐기')          # 예측할 최종 라벨 3개 : 3-class
    # [MULTI-TASK] sub_classes 는 셀 4에서 데이터로부터 자동 발견됩니다.

    # 학습
    img_size: int = 224           # 이미지 크기 224x224
    batch_size: int = 32          # T4 16GB 안정값. OOM 시 16 으로 낮추세요.
    num_workers: int = 0
    epochs: int = 20              # 상한값. 아래 early stopping으로 더 일찍 멈출 수 있음
    lr: float = 1e-4              # 학습률
    weight_decay: float = 1e-3    # [TUNED] 1e-4 → 1e-3 : 과적합 완화를 위해 정규화 강화
    label_smoothing: float = 0.05
    use_amp: bool = True

    # === [TUNED] 과적합 완화용 추가 설정 =========================================
    dropout: float = 0.3          # [TUNED] head 앞 Dropout 비율 (0이면 미사용)
    early_stop_patience: int = 4  # [TUNED] val_F1이 N에폭 연속 개선 없으면 조기 종료
    # 불균형 보정 전략: 'class_weight'(Loss 가중) | 'sampler'(가중 샘플링) | 'none'
    #   원본은 둘 다 켜서 '이중 보정'이었음 → 하나만 쓰도록 정리. ablation 변수로도 활용 가능.
    imbalance_strategy: str = 'class_weight'
    # ==========================================================================

    # 분할 비율 (이미 split 되어 있다면 사용 안 함)
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    seed: int = 42                #  랜덤 시드값

cfg = Config()                    # 위 기본값으로 실제 설정 객체를 만듭니다.
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)      #결과 저장 폴더가 없으면 생성
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2)) #현재 설정값을 json처럼 보기 좋게 출력합니다.

## 3. 데이터 준비 — 매니페스트 빌드
-흩어진 이미지/라벨 파일을 모델 학습용 “엑셀 표 1장”으로 정리하는 단계  
-모델은 폴더 구조를 이해하기 못하니까 모델이 원하는 형식인 이미지 파일 경로, 정답 라벨, 부가 정보를 한 표(dataframe)으로 모읍니다.  
이 표를 manifest 라고 부릅니다.

**확인된 AI-Hub JSON 스키마**

```json
{
  "meta_information": {
    "big_category":    "dispose | recycle | reusable",   ← Main label (3-class)
    "medium_category": "blouse | ...",                    ← Sub label
    "file_name":       "DP_BL_00085.jpg",                 ← 매칭 이미지
    ...
  },
  "annotations": {
    "clothes":    [{"type": ..., "bbox": [...]}],        ← 의류 bbox
    "pollution":  [{"type": ..., "bbox": [...]}],        ← 오염 bbox (XAI GT!)
    "damage":     [{"type": ..., "bbox": [...]}],        ← 손상 bbox (XAI GT!)
    "attachment": [...]
  }
}
```

파일명 prefix(`DP_`/`RC_`/`RU_`)로도 Main 라벨을 알 수 있어 fallback 으로 사용합니다.
- 현실데이터는 종종 라벨 값이 비었거나 오타가 있을 수 있음. 그때 학습 샘플을 버리지 않기 위한 백업 규칙  
파일명 앞 2글자로 라벨 추정:
- DP_ -> 폐기
- RC_ -> 재활용
- RU_ -> 재사용

`pollution` / `damage` bbox 는 **XAI 평가의 GT** 입니다 — §4 의 WSOL 기반
정밀도 평가에서 Grad-CAM 히트맵과 IoU 비교에 그대로 활용됩니다.
지금 단계에선 manifest 에 같이 저장만 해두고, B4 단계에서 사용합니다.

- pollution/damage bbox가 xai GT" 뜻
gt는 ground truth는 정답 기준이라는 의미
B4에서 Grad-cam 히트맵(모델이 주목한 영역)을 만들고, 그 영역이 실제 오염/손상박스와 얼마나 겹치는지(IOU) 계산합니다.  
이때 실제 정답 위치가 바로 pollution/damage bbox입니다. 그래서 지금 단계에서 미리 보관해둡니다.

- B0 분류 학습은 보통 “이미지 전체 + 클래스 라벨”만으로도 가능해서,
이 박스들을 손실 계산에 직접 쓰지 않습니다.
그래도 manifest에 저장하는 이유:
나중 B4 XAI 평가에서 필요하기 때문입니다.

- WSOL, IoU 문장이 의미하는 것  
WSOL: 약지도 위치추정(정확한 위치 라벨 없이도 모델 주목 위치를 보는 평가 맥락)  
IoU: 두 영역 겹침 정도(교집합/합집합)  
번역하면:
“나중에 모델이 본 영역(Grad-CAM)이 진짜 오염/손상 위치와 얼마나 맞는지 수치로 평가한다.”

```
-이 단계 끝나면 실제로 생기는 것
한 행 = 샘플 1개인 표가 생깁니다. 대략 이런 느낌:
file_path: /.../DP_BL_00085.jpg
main: 폐기
sub: blouse
pollution_bbox: [[x1,y1,x2,y2], ...]
damage_bbox: [...]
이 표를 다음 단계(train/val/test 분할)로 넘깁니다.
```

지금 하는 일은 “분류 학습에 필요한 이미지-라벨 매칭 표를 만들고, 미래 XAI 평가용 위치 정답(bbox)까지 함께 적재해 두는 준비 작업”입니다.

In [ ]:
# (로컬 실행) Google Drive 마운트 불필요 — 실행해도 아무 동작 안 합니다.
print('local run: drive mount skipped')


In [ ]:
# 데이터셋 정리 자동화 파이프라인
# 목표: 학습 가능한 표(df)를 안정적으로 만들고 저장

# 1. 상수 정의: 어떤 파일/라벨을 인정할지 규칙 세팅
# 이미지 파일만 골라내기
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# AI-Hub 실제 스키마 (사용자 업로드 샘플로 확인됨)
STATE_MAP = {
    'dispose':  '폐기',
    'recycle':  '재활용',
    'reusable': '재사용',
} #: JSON의 big_category 값을 모델 라벨(한글 3클래스)로 변환

# 파일명 prefix → 상태 (JSON 누락 시 fallback)
# JSON 값이 누락/오류일 때 파일명 prefix로 라벨 복구(fallback).
PREFIX_MAP = {'DP': '폐기', 'RC': '재활용', 'RU': '재사용'}

# 파일명으로 메인 라벨 추론 -> 구별 못할때 fallback용도
# DP_BL_00085.jpg → stem: DP_BL_00085 → 첫 토큰 DP → 폐기 반환.
def state_from_filename(name: str):
    pref = Path(name).stem.split('_')[0].upper()
    return PREFIX_MAP.get(pref)

# Json 1개를 학습용 1행(dict)으로 변환
def parse_label_json(json_path: Path, image_lookup: dict):
    """AI-Hub JSON → dict 또는 None.
    image_lookup: {파일명(stem) → 절대경로} 형태의 사전 빌드 인덱스 (빠른 매칭).
    """
    try: # 오류는 건너뚸서 전체 파이프라인 중단 방지
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        return None

    #핵심 필드 읽기
    meta = data.get('meta_information', {}) #메인 정보 가져와라
    big  = meta.get('big_category', '').strip().lower() # 메인 라벨 추출
    med  = meta.get('medium_category', 'unknown').strip().lower() # 서브 라벨 추출 없으면 unknown
    fname = meta.get('file_name') or f'{json_path.stem}.jpg' #json에 파일명이 없으면 json 이름 기반으로 추정

    # 메인 라벨 확정
    main = STATE_MAP.get(big) or state_from_filename(fname)
    if main is None:
        return None
    # 1순위: big_category / 2순위 : 파일명 prefix fallback / 3순위 : None

    # 이미지 경로 찾기 (인덱스 우선, 없으면 stem 매칭)
    # json이 가리키는 이미지를 실제 파일 시스템에서 찾음. 못찾으면 스킵
    img_path = image_lookup.get(fname) or image_lookup.get(Path(fname).stem)
    if img_path is None:
        return None

    #bbox 및 캡션 추출
    # 여기서 반환되는 dict 1개 = 학습 표의 1행.
    # bbox는 B0 학습에는 직접 안 써도 B4(XAI 평가)용 자산으로 저장.
    ann = data.get('annotations', {})
    return {
        'file_path': str(img_path),
        'main': main,
        'sub':  med,
        # XAI 단계에서 사용할 bbox 들 (없으면 빈 리스트)
        'clothes_bbox':    [b['bbox'] for b in ann.get('clothes', [])],
        'pollution_bbox':  [b['bbox'] for b in ann.get('pollution', [])],
        'damage_bbox':     [b['bbox'] for b in ann.get('damage', [])],
        'attachment_bbox': [b['bbox'] for b in ann.get('attachment', [])],
        'caption': data.get('image_caption', {}).get('caption', ''),
    }

# 전체 폴더를 돌며 최종 DataFrame 생성
def build_manifest(data_root: str) -> pd.DataFrame:
    """
    data_root 아래의 모든 JSON 을 훑어 매니페스트 DataFrame 을 만든다.

    폴더 이름이 무엇이든 (원천데이터/라벨링데이터, images/labels, 또는 그냥 섞여있어도)
    재귀 탐색으로 동작한다.
    """
    data_root = Path(data_root)

    #경로 존재하는지 유무
    if not data_root.exists():
        raise FileNotFoundError(f"data_root not found: {data_root}")

    # 1) 이미지 인덱스 빌드 (한 번만 rglob)
    # rglob로 재귀 탐색으로 깊은 곳까지 검색
    # 파일명과 stem 둘 다 키로 넣어 매칭 성공률을 높임.
    # JSON마다 매번 파일 찾지 않고, 한 번 인덱싱 후 빠르게 조회
    print('Indexing images...', flush=True)
    image_lookup = {}
    for p in data_root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            image_lookup[p.name] = p                  # 파일명 전체
            image_lookup.setdefault(p.stem, p)        # stem only
    print(f'  {len(image_lookup)} image files indexed')

    # 2) JSON 순회
    # 유효한것들을 rows 안에 누적 / 실패한거는 skipped 집계.
    print('Parsing JSON labels...', flush=True)
    rows, skipped = [], 0
    for jp in data_root.rglob('*.json'):
        r = parse_label_json(jp, image_lookup)
        if r is None:
            skipped += 1
            continue
        rows.append(r)
    # dataFrame 정리 / 동일 이미지 중복 라벨 제거, 인덱스 재정렬
    # 학습 시 중복 샘플로 분포가 왜곡되는 문제 방지.
    df = pd.DataFrame(rows).drop_duplicates(subset='file_path').reset_index(drop=True)
    print(f'  parsed {len(df)} samples, skipped {skipped}')
    return df

# df 생성 직후 검증 및 요약 출력 코드
# 데이터가 0개면 바로 중단해서 에러 방지
df = build_manifest(cfg.data_root)
assert len(df) > 0, '매니페스트가 비었습니다. cfg.data_root 경로 확인 필요.'
# 미리보기 (bbox 제외)
display(df.drop(columns=[c for c in df.columns if c.endswith('_bbox')]).head())
# 클래스 불균형 상태 확인 (중요) -> 이후에 weightedRandomSampler, class weight 필요성 판단 근거.
print('\n[Main] 클래스 분포'); print(df['main'].value_counts())
print('\n[Sub]  클래스 분포 (상위 15)'); print(df['sub'].value_counts().head(15))
# XAI GT로 쓸 bbox가 실제 얼마나 있는지 체크 -> 데이터 품질 진단 리포트 역할
print('\n오염 bbox 보유 샘플:', (df['pollution_bbox'].apply(len) > 0).sum())
print('손상 bbox 보유 샘플:', (df['damage_bbox'].apply(len) > 0).sum())

# 매니페스트 저장 (XAI 단계에서 재사용)
# parquet로 빠르고 용량 효율 좋음.
# 안되면 csv로 자동 fallback
# 덕분에 다음 실행 때 JSON 다시 뒤지지 않아도 됨.
manifest_path = Path(cfg.output_dir) / 'manifest.parquet'
try:
    df.to_parquet(manifest_path)
    print('\nSaved manifest →', manifest_path)
except Exception:
    manifest_path = Path(cfg.output_dir) / 'manifest.csv'
    df.to_csv(manifest_path, index=False)
    print('\nSaved manifest (csv) →', manifest_path)


## 4. Train / Val / Test 분할 (Stratified)

In [ ]:
# 알 수 없는 main 라벨은 제외
#(재사용, 재활용, 폐기 중 하나인 샘플만 남깁니다.)
df = df[df['main'].isin(cfg.main_classes)].reset_index(drop=True)

# 메인 라벨을 숫자로 변환
# 문자열 라벨을 모델이 쓰기 쉬운 정수로 매핑
# (순서는 cfg.main_classes 순서에 따름)
main2idx = {c: i for i, c in enumerate(cfg.main_classes)}
df['main_idx'] = df['main'].map(main2idx)

# [MULTI-TASK] sub_classes 자동 발견 — 빈도순 정렬
# 목적:
# 지금(B0)은 주로 main_idx를 쓰지만, 나중 멀티태스크(B2)에서 sub_idx를 바로 사용할 수 있게 준비.
SUB_CLASSES = tuple(df['sub'].value_counts().index.tolist())
sub2idx = {c: i for i, c in enumerate(SUB_CLASSES)}
df['sub_idx'] = df['sub'].map(sub2idx)
#서브 클래스 목록 출력
print(f'Discovered {len(SUB_CLASSES)} sub_classes: {SUB_CLASSES}')

# 1차 분할 : train vs temp
# val + test를 합친 비율(여기선 0.30)을 먼저 떼어 temp로 둡니다.
# stratify=df['main_idx']로 클래스 비율 유지.
# random_state=seed로 재현 가능.
# 현재 설정 기준 비율: train 70%  // temp 30%
train_df, temp_df = train_test_split(
    df, test_size=cfg.val_ratio + cfg.test_ratio,
    stratify=df['main_idx'], random_state=cfg.seed,
)

# 2차 분할: temp -> val/ test
#temp 내부에서 val:test 비율을 맞춰 나눔.
#현재 val=0.15, test=0.15라 temp 안에서는 50:50.
#결과적으로 전체 기준 70/15/15가 됩니다.
#역시 stratify로 클래스 비율 유지.
val_ratio_in_temp = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
val_df, test_df = train_test_split(
    temp_df, test_size=1 - val_ratio_in_temp,
    stratify=temp_df['main_idx'], random_state=cfg.seed,
)
# 분할 크기 출력
print(f'\nTrain {len(train_df):>6} | Val {len(val_df):>6} | Test {len(test_df):>6}')


# === [LOCAL FAIR-SPLIT 저장] ===
import os as _os
_split_dir = f'{RUNS_ROOT}/_split'
_os.makedirs(_split_dir, exist_ok=True)
train_df[['file_path']].to_csv(_os.path.join(_split_dir,'train_files.csv'), index=False)
val_df[['file_path']].to_csv(_os.path.join(_split_dir,'val_files.csv'), index=False)
test_df[['file_path']].to_csv(_os.path.join(_split_dir,'test_files.csv'), index=False)
print(f'[split 저장 완료] -> {_split_dir}')


## 5. Augmentation — 물리적 타당성 반영

**적용**
- `RandomResizedCrop`: 컨베이어 벨트 다양한 위치/스케일
- `RandomHorizontalFlip`: 좌우 대칭은 의류에 자연스러움
- `ColorJitter`: 선별장 조명 변화
- `GaussianBlur` (확률 낮게): 카메라 흔들림/먼지

**의도적 배제**
- `RandomVerticalFlip` ❌
  - 상하 반전은 옷의 위/아래(목/허리/밑단)를 뒤집어 **종류(category) 분류를 방해**.
    제안서 §3 의 *데이터 이해도* 증명 포인트.


In [ ]:
MEAN = [0.485, 0.456, 0.406]   # ImageNet 통계 (사전학습 정렬)
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([ # compose는 안에 있는걸 전부 합쳐서 사용
    transforms.Resize((cfg.img_size + 32, cfg.img_size + 32)),
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(p=0.5),   # ❌ 의도적 배제 (위 markdown 참조)
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


## 6. Dataset & DataLoader (+ 클래스 불균형 보정)

In [ ]:
from PIL import Image

class ClothingDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['file_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        # [MULTI-TASK] 두번째 라벨도 함께 반환하려면 (img, main_idx, sub_idx) 형태로
        return img, int(row['main_idx'])

train_ds = ClothingDataset(train_df, train_tf)
val_ds   = ClothingDataset(val_df,   eval_tf)
test_ds  = ClothingDataset(test_df,  eval_tf)

# 클래스 카운트 / Loss용 class_weight 계산
class_counts = train_df['main_idx'].value_counts().sort_index().values
class_weights = torch.tensor(class_counts.sum() / (len(class_counts) * class_counts),
                             dtype=torch.float32)

print('클래스 카운트:', dict(zip(cfg.main_classes, class_counts.tolist())))
print('Loss class_weights:', class_weights.tolist())

# === [TRUNCATED IMAGE FIX] 각 DataLoader 워커에도 동일 플래그 적용 ===
def _worker_init(worker_id):
    from PIL import ImageFile
    ImageFile.LOAD_TRUNCATED_IMAGES = True

# === [TUNED] 이중 불균형 보정 정리 ===========================================
# 원본은 WeightedRandomSampler(샘플링)와 Loss class_weight를 '동시에' 사용 →
# 같은 보정이 두 번 걸려 소수 클래스(폐기)에 과하게 쏠릴 수 있었음.
# 여기서는 cfg.imbalance_strategy 로 하나만 선택한다.
#   - 'sampler'       : 가중 샘플링만 사용 (Loss는 class_weight 미적용 → 셀 8 참조)
#   - 'class_weight'  : Loss 가중만 사용 (일반 shuffle 샘플링)  ← 기본값/권장
#   - 'none'          : 둘 다 미사용
if cfg.imbalance_strategy == 'sampler':
    sample_weights = train_df['main_idx'].map(
        {i: float(w) for i, w in enumerate(1.0 / class_counts)}
    ).values
    sampler = WeightedRandomSampler(sample_weights,
                                    num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler,
                              num_workers=cfg.num_workers, pin_memory=True,
                              worker_init_fn=_worker_init)
    print("불균형 보정: WeightedRandomSampler 사용 (Loss class_weight는 미적용)")
else:
    # 'class_weight' 또는 'none' → 일반 shuffle 로더
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True,
                              worker_init_fn=_worker_init)
    print(f"불균형 보정 전략: {cfg.imbalance_strategy} (shuffle 로더 사용)")
# ==========================================================================

val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True, worker_init_fn=_worker_init)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True, worker_init_fn=_worker_init)

## 7. 모델 — Plain ResNet-50

> Strict Baseline: 사전학습된 ResNet-50의 `fc` 만 3-class 로 교체.
> Multi-Task 확장 지점은 `# [MULTI-TASK]` 주석으로 표시.

In [ ]:
class PlainResNet(nn.Module):
    def __init__(self, num_main: int = 3,
                 # [MULTI-TASK] num_sub: int = 11,
                 pretrained: bool = True,
                 dropout: float = 0.0):   # [TUNED] head 앞 Dropout 비율
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=weights)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()           # feature extractor
        self.dropout = nn.Dropout(p=dropout)       # [TUNED] 과적합 완화 (p=0이면 효과 없음)
        self.head_main = nn.Linear(in_feat, num_main)
        # [MULTI-TASK] self.head_sub = nn.Linear(in_feat, num_sub)

    def forward(self, x):
        f = self.backbone(x)                       # (B, 2048) — 이미 GAP 적용됨
        f = self.dropout(f)                        # [TUNED]
        out_main = self.head_main(f)
        # [MULTI-TASK] out_sub = self.head_sub(f); return out_main, out_sub
        return out_main

model = PlainResNet(num_main=len(cfg.main_classes), dropout=cfg.dropout).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params/1e6:.2f} M  | Dropout p={cfg.dropout}')

## 8. 학습 루프 (AMP + Cosine + Best-checkpoint)

In [ ]:
# === [TUNED] 불균형 보정 전략에 맞춰 Loss의 class_weight 적용 여부 결정 ===
# 'sampler' 전략이면 샘플링에서 이미 보정했으므로 Loss 가중은 끈다 (이중 보정 방지).
if cfg.imbalance_strategy == 'class_weight':
    _loss_weight = class_weights.to(DEVICE)
    print("CrossEntropyLoss: class_weight 적용")
else:
    _loss_weight = None
    print("CrossEntropyLoss: class_weight 미적용")

criterion = nn.CrossEntropyLoss(
    weight=_loss_weight, label_smoothing=cfg.label_smoothing
)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                              weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=cfg.lr * 0.01,
)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)  # 나중 버전: torch.amp.GradScaler('cuda', enabled=cfg.use_amp)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total, correct = 0.0, 0, 0
    all_y, all_p = [], []
    for imgs, ys in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        ys   = ys.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.use_amp): # 나중 버전: torch.amp.autocast('cuda', enabled=cfg.use_amp)
            logits = model(imgs)
            loss = criterion(logits, ys)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        correct += (preds == ys).sum().item(); total += imgs.size(0)
        all_y += ys.cpu().tolist(); all_p += preds.cpu().tolist()
    p, r, f1, _ = precision_recall_fscore_support(all_y, all_p,
                                                  average='macro', zero_division=0)
    return total_loss/total, correct/total, p, r, f1


history = []
best_f1, best_state, best_epoch = -1, None, 0
ckpt = Path(cfg.output_dir) / 'best.pt'

# === [TUNED] Early stopping 상태 변수 ===
epochs_no_improve = 0   # val_F1이 개선되지 않은 연속 에폭 수

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc, _, _, tr_f1 = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va_loss, va_acc, va_p, va_r, va_f1 = run_epoch(val_loader, train=False)
    scheduler.step()
    history.append(dict(epoch=epoch, tr_loss=tr_loss, va_loss=va_loss,
                        tr_acc=tr_acc, va_acc=va_acc, va_f1=va_f1))
    print(f'[{epoch:02d}/{cfg.epochs}] '
          f'loss {tr_loss:.3f}/{va_loss:.3f}  '
          f'acc {tr_acc:.3f}/{va_acc:.3f}  '
          f'val_F1 {va_f1:.3f}  ({time.time()-t0:.1f}s)')

    # === [TUNED] best 갱신 + early stopping 판정 ===
    if va_f1 > best_f1:
        best_f1 = va_f1
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save({'model': best_state, 'cfg': asdict(cfg),
                    'val_f1': best_f1, 'epoch': epoch}, ckpt)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= cfg.early_stop_patience:
            print(f'⏹ Early stopping: val_F1이 {cfg.early_stop_patience}에폭 연속 개선 없음 '
                  f'(best epoch {best_epoch}, F1 {best_f1:.4f})')
            break

print(f'\n✅ Best val F1-macro: {best_f1:.4f}  (epoch {best_epoch}, saved → {ckpt})')
hist_df = pd.DataFrame(history)
hist_df.to_csv(Path(cfg.output_dir) / 'history.csv', index=False)

In [ ]:
# 학습 곡선
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist_df.epoch, hist_df.tr_loss, label='train'); ax[0].plot(hist_df.epoch, hist_df.va_loss, label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist_df.epoch, hist_df.tr_acc, label='train_acc')
ax[1].plot(hist_df.epoch, hist_df.va_acc, label='val_acc')
ax[1].plot(hist_df.epoch, hist_df.va_f1,  label='val_F1', linestyle='--')
ax[1].set_title('Acc / F1'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig(Path(cfg.output_dir)/'curves.png', dpi=120); plt.show()


## 9. 테스트 평가 — Classification Report & Confusion Matrix

In [ ]:
# best checkpoint 로 복원
# model.load_state_dict(best_state)   ← 이 줄 대신
ckpt = torch.load(Path(cfg.output_dir)/'best.pt', map_location=DEVICE)
model.load_state_dict(ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt)
model.eval()

all_y, all_p, all_prob = [], [], []
with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.use_amp):
    for imgs, ys in test_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        logits = model(imgs)
        prob = torch.softmax(logits, 1).cpu().numpy()
        all_prob.append(prob)
        all_p += logits.argmax(1).cpu().tolist()
        all_y += ys.tolist()
all_prob = np.concatenate(all_prob)

print(classification_report(all_y, all_p, target_names=cfg.main_classes, digits=4))
acc = accuracy_score(all_y, all_p)
p, r, f1, _ = precision_recall_fscore_support(all_y, all_p, average='macro', zero_division=0)
pw, rw, f1w, _ = precision_recall_fscore_support(all_y, all_p, average='weighted', zero_division=0)
print(f'Accuracy        : {acc:.4f}')
print(f'Macro P/R/F1    : {p:.4f} / {r:.4f} / {f1:.4f}')
print(f'Weighted P/R/F1 : {pw:.4f} / {rw:.4f} / {f1w:.4f}')

metrics = dict(accuracy=acc, macro_f1=f1, weighted_f1=f1w,
               macro_precision=p, macro_recall=r,
               weighted_precision=pw, weighted_recall=rw)
with open(Path(cfg.output_dir)/'test_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Windows 기본 한글 폰트: 맑은 고딕
font_path = r"C:/Windows/Fonts/malgun.ttf"
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("Font exists:", os.path.exists(font_path))
print("Current font:", plt.rcParams["font.family"])

In [ ]:
cm = confusion_matrix(all_y, all_p)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cfg.main_classes, yticklabels=cfg.main_classes, ax=ax[0])
ax[0].set_title('Confusion Matrix (count)')
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=cfg.main_classes, yticklabels=cfg.main_classes, ax=ax[1])
ax[1].set_title('Confusion Matrix (row-normalized)')
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
plt.tight_layout(); plt.savefig(Path(cfg.output_dir)/'confusion_matrix.png', dpi=120); plt.show()


## 10. 효율성 지표 — Params / FLOPs / FPS

제안서 §3 의 *현장 적용 가능성(12GB VRAM)* 평가 포인트.
이후 단계 모델과 동일한 입력 크기/배치로 비교해야 의미가 있습니다.

In [ ]:
from thop import profile

# Params
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# FLOPs (입력 1장 기준)
dummy = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=DEVICE)
with torch.no_grad():
    flops, _ = profile(model, inputs=(dummy,), verbose=False)

# FPS (배치=1, 워밍업 후 측정)
model.eval()
with torch.no_grad():
    for _ in range(10): _ = model(dummy)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    N = 100
    for _ in range(N): _ = model(dummy)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    fps = N / (time.time() - t0)

# Peak GPU memory (MB)
mem_mb = (torch.cuda.max_memory_allocated() / 1024**2) if DEVICE.type == 'cuda' else 0.0

eff = dict(params_M=round(n_params/1e6, 3),
           trainable_M=round(n_trainable/1e6, 3),
           gflops=round(flops/1e9, 3),
           fps_batch1=round(fps, 2),
           peak_gpu_mem_MB=round(mem_mb, 1))
print(json.dumps(eff, indent=2))
with open(Path(cfg.output_dir)/'efficiency.json', 'w') as f:
    json.dump(eff, f, indent=2)


## 11. 다음 단계 (Ablation 로드맵)

본 베이스라인 산출물은 모두 `cfg.output_dir` 아래 저장되어 있습니다.
이후 실험을 동일한 형식으로 저장하면 한 장의 표로 비교 가능합니다.

| Step | 변경점 | 가설 | 측정 지표 |
|---|---|---|---|
| ✅ **B0** *(본 노트북)* | Plain ResNet-50 + 단일 헤드 | 기준점 | F1-macro, Params, FLOPs, FPS |
| B1 | + Texture-aware Block (앞단 Conv, stride 1) | 미세 질감 보존 → '재활용/폐기' 구분 ↑ | Confusion Matrix의 (재활용↔폐기) 오분류 감소 |
| B2 | + Multi-Task Head (Main 3-cls + Sub 11-cls) | 종류 정보가 상태 분류의 보조 신호 | Main F1 ↑ + Sub F1 동시 보고 |
| B3 | Backbone → DenseNet-121 | Feature reuse로 미세 질감 끝까지 전달 | F1-macro ↑, Params ↓ |
| B4 | + Grad-CAM / WSOL (IoU) | XAI: 오염 부위에 활성화되는가 | 정성 시각화 + IoU 비교 |

저장된 파일
- `best.pt` — 모델 체크포인트
- `history.csv`, `curves.png` — 학습 곡선
- `test_metrics.json` — F1/Precision/Recall (macro & weighted)
- `confusion_matrix.png` — 오분류 패턴
- `efficiency.json` — Params / FLOPs / FPS / Peak GPU mem
